In [ ]:
# ----Inputs/Output----
EMBED_TIFF = "/path/to/original/embeddings"        
REF_RASTER_PATH = "/path/to/a/aligned/reference/for/CRS/transform/size" 
OUT_DIR = "/path/to/output/directory" 

K = 32
RANDOM_STATE = 42
TEMPERATURE = 1.0
BATCH_SIZE = 8192
MAX_ITER = 100

In [ ]:
import os
import math
import numpy as np
from pathlib import Path

import rasterio
from rasterio.enums import Resampling

from sklearn.cluster import MiniBatchKMeans
from sklearn.utils import shuffle

In [ ]:
def read_embedding_geotiff(tif_path):
    with rasterio.open(tif_path) as src:
        data = src.read()  
    return np.transpose(data, (1,2,0)) 

def read_ref_profile(ref_path):
    with rasterio.open(ref_path) as src:
        profile = src.profile.copy()
        width, height = src.width, src.height
        transform = src.transform
        crs = src.crs
    return profile, width, height, transform, crs

def ensure_hw_match(arr, ref_hw):
    H, W = arr.shape[:2]
    rH, rW = ref_hw
    if (H, W) == (rH, rW):
        return arr
    print(f"[warn] Embedding size {H}x{W} != reference {rH}x{rW}. Resampling to match reference.")
    out = np.empty((rH, rW, arr.shape[2]), dtype=arr.dtype)
    for c in range(arr.shape[2]):
        band = arr[..., c]
        y_idx = (np.linspace(0, H-1, rH)).astype(int)
        x_idx = (np.linspace(0, W-1, rW)).astype(int)
        out[..., c] = band[np.ix_(y_idx, x_idx)]
    return out

def write_geotiff_singleband(path, data, base_profile, force_dtype=None, nodata_policy="auto"):
    import numpy as np
    prof = base_profile.copy()
    prof.update(driver="GTiff", count=1, compress="LZW", tiled=True)

    if force_dtype is not None:
        prof["dtype"] = np.dtype(force_dtype).name
        data = data.astype(force_dtype, copy=False)
    else:
        prof["dtype"] = np.dtype(data.dtype).name

    if isinstance(nodata_policy, (int, float)):
        prof["nodata"] = nodata_policy
    elif nodata_policy == "auto":
        dt = np.dtype(prof["dtype"])
        if np.issubdtype(dt, np.integer):
            prof["nodata"] = 0
            if np.issubdtype(data.dtype, np.floating):
                data = np.where(np.isnan(data), 0, data).astype(dt, copy=False)
        else:
            prof.pop("nodata", None)
    elif nodata_policy is None:
        prof.pop("nodata", None)
    else:
        raise ValueError("Unknown nodata_policy")

    with rasterio.open(path, "w", **prof) as dst:
        dst.write(data[np.newaxis, ...])

In [ ]:
def fit_kmeans(emb, k=32, random_state=42, max_iter=100, batch_size=8192):
    H, W, C = emb.shape
    n_pixels = H * W
    n_sample = min(500000, n_pixels)
    flat = emb.reshape(-1, C)
    if n_sample < n_pixels:
        sample = shuffle(flat, random_state=random_state, n_samples=n_sample)
    else:
        sample = flat
    km = MiniBatchKMeans(n_clusters=k, random_state=random_state, batch_size=batch_size, max_iter=max_iter, verbose=0)
    km.fit(sample)
    return km

def soft_assignment_from_dist(dist, temperature=1.0, eps=1e-9):
    x = -dist / max(temperature, eps)
    x = x - x.min(axis=1, keepdims=True)
    ex = np.exp(x)
    probs = ex / (ex.sum(axis=1, keepdims=True) + eps)
    return probs

def cluster_and_probs(emb, km, temperature=1.0):
    H, W, C = emb.shape
    flat = emb.reshape(-1, C)
    centers = km.cluster_centers_.astype(flat.dtype, copy=False)
    x2 = (flat**2).sum(axis=1, keepdims=True)             
    c2 = (centers**2).sum(axis=1)[np.newaxis, ...]        
    xmu = flat @ centers.T                                
    dist2 = np.maximum(x2 + c2 - 2.0 * xmu, 0.0)          
    dist = np.sqrt(dist2, dtype=np.float32)               

    probs = soft_assignment_from_dist(dist, temperature=temperature)  
    labels_0based = np.argmax(probs, axis=1).astype(np.int32)         
    max_prob = probs[np.arange(probs.shape[0]), labels_0based]        

    import math
    with np.errstate(divide='ignore', invalid='ignore'):
        entropy = -np.nansum(probs * np.log(probs + 1e-12), axis=1)
    norm_entropy = entropy / math.log(probs.shape[1])
    stability = 1.0 - norm_entropy
    stability = np.clip(stability, 0.0, 1.0)

    labels_1based = (labels_0based + 1).astype(np.uint8)

    labels_1based = labels_1based.reshape(H, W)
    max_prob = max_prob.reshape(H, W).astype(np.float32)
    stability = stability.reshape(H, W).astype(np.float32)
    return labels_1based, max_prob, stability

In [ ]:
from pathlib import Path
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

ref_profile, ref_w, ref_h, ref_transform, ref_crs = read_ref_profile(REF_RASTER_PATH)
ref_profile.update(width=ref_w, height=ref_h, transform=ref_transform, crs=ref_crs)

emb = read_embedding_geotiff(EMBED_TIFF)
emb = ensure_hw_match(emb, (ref_h, ref_w))
print("Embedding shape (H,W,C):", emb.shape)

km = fit_kmeans(emb, k=K, random_state=RANDOM_STATE, max_iter=MAX_ITER, batch_size=BATCH_SIZE)
labels_1based, max_prob, stability = cluster_and_probs(emb, km, temperature=TEMPERATURE)

labels_path = str(Path(OUT_DIR) / "labels_k32.tif")
maxprob_path = str(Path(OUT_DIR) / "maxprob.tif")
stability_path = str(Path(OUT_DIR) / "stability.tif")

labels_profile = ref_profile.copy()
labels_profile.update(dtype="uint8", nodata=0)
write_geotiff_singleband(labels_path, labels_1based.astype(np.uint8, copy=False), labels_profile, force_dtype="uint8", nodata_policy="auto")

float_profile = ref_profile.copy()
float_profile.update(dtype="float32")
float_profile.pop("nodata", None)

write_geotiff_singleband(maxprob_path, max_prob.astype(np.float32, copy=False), float_profile, force_dtype="float32", nodata_policy="auto")
write_geotiff_singleband(stability_path, stability.astype(np.float32, copy=False), float_profile, force_dtype="float32", nodata_policy="auto")

print("Done. Written files:")
print(" -", labels_path)
print(" -", maxprob_path)
print(" -", stability_path)